## Data Preprocessing

In [1]:
import pandas as pd

In [2]:
# read in csv file 
df = pd.read_csv("../data/raw/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv")

In [34]:
# rename columns
column_to_rename_list = ["flat_type", "flat_model", "lease_commence_date"]
new_col_name_list = ["no_of_rooms_structure", "building_type", "lease_commence_year"]

mapping = dict(zip(column_to_rename_list, new_col_name_list))
missing_cols = [col for col in column_to_rename_list if col not in df.columns]

df = df.rename(columns=mapping)

In [35]:
# convert remaining_lease values to individual columns of years and months
df[['remaining_lease_years', 'remaining_lease_months']] = df['remaining_lease'].str.split(' years', expand=True)

# remove text
df['remaining_lease_months'] = (
    df['remaining_lease_months']
    .str.replace('months', '', regex=False)
    .str.replace('month', '', regex=False)
    .str.strip() 
)

# convert string to int for years and months
df[['remaining_lease_years', 'remaining_lease_months']] = (
    df[['remaining_lease_years', 'remaining_lease_months']]
    .replace('', 0)
    .astype(int)
)

# create another column with the accurate number of years and months rounded to 2 dp
df['remaining_lease_exact_years'] = round((df['remaining_lease_months'] / 12.0) + df['remaining_lease_years'], 2)

In [36]:
# convert resale price to int
df['resale_price'] = df['resale_price'].astype(int)

In [37]:
# convert column to datetime and then to year
df["lease_commence_year"] = pd.to_datetime(df["lease_commence_year"], format='%Y').dt.year

In [38]:
# split registration year into separate columns for year and month
df[['rego_year', 'rego_month']] = df['month'].str.split('-', expand=True)
df[['rego_year', 'rego_month']] = df[['rego_year', 'rego_month']].astype(int)

In [39]:
# drop the original remaining_lease and month columns
df = df.drop(columns=['month', 'remaining_lease'])

In [42]:
# save the cleaned df as a csv 
df.to_csv('../data/processed/cleaned_hdb_resale_price.csv', index=False)